# ArgumentCorrectnessMetric

## What it measures

Whether the *arguments* the agent passed to each tool were appropriate for the request. The
judge reads `input` and every entry in `tools_called` and assesses whether the
`input_parameters` are correct and sufficient for what was asked.

It is the natural complement to `ToolCorrectnessMetric`. That metric asks whether the right
tools were reached for; this one asks whether they were reached for *competently*. An agent
that calls the sanctions screen with an empty name, the wrong party's name, or a company's
incorporation date in the `dob` field scores well on tool correctness and badly here.

## When it is useful

On agents whose tools take semantically meaningful arguments rather than free text -
screening a specific named party, checking a specific ISO country code, fetching evidence
for a specific case. It catches the argument-construction bugs that produce a technically
successful tool call with a meaningless result: `match_count: 0` because the wrong name was
screened is indistinguishable from a genuine clean screen unless someone checks the input.

## DeepEval inputs and test-case type

| DeepEval field | Required |
|---|---|
| test case type | `LLMTestCase` |
| `input` | yes |
| `tools_called` | yes - `list[ToolCall]`, with `input_parameters` populated |

No `expected_tools`, and no golden. The judge reasons about appropriateness from the task
statement and the calls themselves.

In [ ]:
# --------------------------------------------------------------------------
# Configuration. Every value comes from the environment - nothing about this
# machine, this port or this deployment is baked into the notebook.
# --------------------------------------------------------------------------
import json
import os
import textwrap
from pathlib import Path

import httpx
from dotenv import load_dotenv

# Look for .env next to the notebook, then one level up (the project root).
for _candidate in (Path.cwd() / ".env", Path.cwd().parent / ".env"):
    if _candidate.is_file():
        load_dotenv(_candidate)
        break


class MissingConfiguration(RuntimeError):
    """Raised when a required environment variable is absent."""


def env(name, default=None, *, required=False):
    value = os.environ.get(name) or default
    if required and not value:
        raise MissingConfiguration(
            f"Environment variable {name!r} is not set.\n"
            f"Copy .env.example to .env and fill it in, or export {name} before "
            f"starting the kernel. See README.md -> '.env configuration'."
        )
    return value


API_BASE = env("AML_API_BASE_URL", "http://localhost:8000").rstrip("/")
API_TIMEOUT_S = float(env("AML_API_TIMEOUT_S", "180"))
EXPECTED_SEED_VERSION = env("AML_EXPECTED_SEED_VERSION", "scenarios-v1")
RESET_BEFORE_RUN = env("AML_RESET_BEFORE_RUN", "false").lower() in ("1", "true", "yes")

# Every notebook needs an OpenAI key. ToolCorrectnessMetric scores without any
# LLM call, but DeepEval 4.1.4 still builds a GPTModel in its constructor and
# raises without a key, so the key is required there too - just never used.
JUDGE_MODEL = env("DEEPEVAL_JUDGE_MODEL", "gpt-5.4-mini")
os.environ.setdefault("DEEPEVAL_TELEMETRY_OPT_OUT", "YES")

# One key per role, falling back to the single AML_API_KEY. Blank is correct
# when the application runs with AUTH_MODE=off (its default).
API_KEYS = {
    "analyst": env("AML_API_KEY_ANALYST") or env("AML_API_KEY", ""),
    "eval_reader": env("AML_API_KEY_EVAL_READER") or env("AML_API_KEY", ""),
    "test_operator": env("AML_API_KEY_TEST_OPERATOR") or env("AML_API_KEY", ""),
}

print(f"API base URL       : {API_BASE}")
print(f"Request timeout    : {API_TIMEOUT_S}s")
print(f"Judge model        : {JUDGE_MODEL}")
print(f"Expected seed      : {EXPECTED_SEED_VERSION}")
print(f"API key configured : {bool(API_KEYS['analyst'])}  (False is correct when AUTH_MODE=off)")
print(f"OPENAI_API_KEY set : {bool(os.environ.get('OPENAI_API_KEY'))}")

In [ ]:
# --------------------------------------------------------------------------
# A small HTTP client. Every failure mode the application can present is
# turned into a message that names the cause and the thing to check.
# --------------------------------------------------------------------------
# The contract these notebooks were written against. The application may
# serve a HIGHER minor version: a MINOR bump is additive by its own
# contract policy (1.0.0 -> 1.1.0 added HealthResponse.build_version and
# changed nothing else), so treating it as a mismatch would turn this
# guard into noise on every single call. Only a MAJOR change, or an
# application older than these notebooks, is a problem.
EXPECTED_SCHEMA_VERSION = "1.0.0"


def contract_version(value):
    """(major, minor) from a MAJOR.MINOR.PATCH string, or None."""
    try:
        parts = value.split("+")[0].split(".")
        return int(parts[0]), int(parts[1])
    except (AttributeError, IndexError, ValueError):
        return None

SECRET_KEY_HINTS = ("api_key", "apikey", "authorization", "secret", "password",
                    "credential", "token")


def redact(value):
    """Mask credential-like values before anything is printed."""
    if isinstance(value, dict):
        return {
            k: ("***REDACTED***" if any(h in k.lower() for h in SECRET_KEY_HINTS)
                else redact(v))
            for k, v in value.items()
        }
    if isinstance(value, list):
        return [redact(v) for v in value]
    return value


class ApiError(RuntimeError):
    """A non-2xx response, carrying the application's error envelope."""


def api(method, path, *, role="analyst", json_body=None, params=None,
        expect_status=None):
    """Call the application API and return parsed JSON.

    role selects which API key is sent. It only matters when the application
    runs with AUTH_MODE=api_key; with AUTH_MODE=off the header is omitted.
    """
    headers = {"Accept": "application/json"}
    key = API_KEYS.get(role, "")
    if key:
        headers["X-API-Key"] = key

    url = f"{API_BASE}{path}"
    try:
        response = httpx.request(method, url, headers=headers, json=json_body,
                                 params=params, timeout=API_TIMEOUT_S)
    except httpx.ConnectError as exc:
        raise ApiError(
            f"Could not connect to {url}.\n"
            f"  - Is the application running?  curl {API_BASE}/api/health\n"
            f"  - Is AML_API_BASE_URL correct? It is currently {API_BASE!r}.\n"
            f"  - Underlying error: {exc}"
        ) from exc
    except httpx.TimeoutException as exc:
        raise ApiError(
            f"{method} {url} timed out after {API_TIMEOUT_S}s.\n"
            f"  - An investigation run does retrieval, several MCP tool calls and\n"
            f"    one LLM synthesis; raise AML_API_TIMEOUT_S if this is expected.\n"
            f"  - Underlying error: {exc!r}"
        ) from exc

    served = response.headers.get("X-Schema-Version")
    served_version = contract_version(served) if served else None
    expected_version = contract_version(EXPECTED_SCHEMA_VERSION)
    if served_version and served_version[0] != expected_version[0]:
        raise ApiError(
            f"The application serves contract version {served}; these notebooks were "
            f"written against {EXPECTED_SCHEMA_VERSION}. A MAJOR change means fields "
            f"may have been removed or retyped - re-derive the goldens against the "
            f"new contract rather than scoring against one they do not match."
        )
    if served_version and served_version[1] < expected_version[1]:
        print(f"WARNING: application reports contract version {served}, older than "
              f"the {EXPECTED_SCHEMA_VERSION} these notebooks were written against. "
              f"Fields the goldens rely on may not exist yet.")

    if response.status_code >= 400:
        try:
            envelope = response.json()
        except ValueError:
            envelope = {"raw_body": response.text[:1000]}
        hint = {
            401: "AUTH_MODE=api_key is on and no valid X-API-Key was sent. Set AML_API_KEY.",
            403: "The key's role may not reach this endpoint. eval_reader is needed for "
                 "/api/agent/trace and /api/eval/*; test_operator for /api/dev/reset and "
                 "/api/mcp/invoke.",
            404: "The id does not exist. Resolve ids from GET /api/eval/scenarios rather "
                 "than hardcoding them.",
            409: "Often index_not_built - the vector index has never been built. "
                 "POST /api/dev/reset once, or set AML_RESET_BEFORE_RUN=true.",
            502: "The application's LLM provider failed or returned output that broke its "
                 "own schema contract. Retry, or inspect GET /api/agent/trace/{run_id}.",
            503: "llm_not_configured - the application has no OPENROUTER_API_KEY. "
                 "This is the application's key, not the judge's OPENAI_API_KEY.",
        }.get(response.status_code, "")
        raise ApiError(
            f"{method} {url} -> HTTP {response.status_code}\n"
            f"  envelope: {json.dumps(envelope, indent=2)[:1200]}\n"
            + (f"  hint: {hint}" if hint else "")
        )

    if expect_status is not None and response.status_code != expect_status:
        raise ApiError(f"{method} {url} -> expected HTTP {expect_status}, "
                       f"got {response.status_code}")

    if not response.content:
        return None
    try:
        return response.json()
    except ValueError as exc:
        raise ApiError(
            f"{method} {url} returned HTTP {response.status_code} but the body is not "
            f"JSON.\n  first 500 bytes: {response.text[:500]!r}"
        ) from exc


def show(title, payload, limit=2500):
    """Pretty-print a payload with secrets masked and long bodies truncated."""
    text = json.dumps(redact(payload), indent=2, default=str)
    print(f"----- {title} -----")
    print(text if len(text) <= limit else text[:limit] + f"\n... [{len(text) - limit} more characters]")


health = api("GET", "/api/health")
show("GET /api/health", health)
if not health.get("status") == "ok":
    raise ApiError(f"Application is not healthy: {health}")

## Endpoints exercised

| Endpoint | Role here |
|---|---|
| `POST /api/cases/{case_id}/investigate` | Runs the agent |
| `GET /api/eval/export/{run_id}` | Supplies `tools_called` already in DeepEval's `ToolCall` field names |
| `GET /api/mcp/tools` | Live JSON Schema per tool, for a deterministic pre-check |

Scenario **s3, "Possible sanctions name match (beneficiary near-miss)"** is used because
its argument construction is genuinely non-trivial: there is a business customer *and* an
individual beneficiary, so the agent must screen two different parties, attach a date of
birth to one of them and not the other, and check the beneficiary's country rather than the
customer's. Several distinct argument mistakes are available to be made.

In [ ]:
# --------------------------------------------------------------------------
# Resolve scenarios to live row ids. Seed ids are assigned by insert order, so
# a hardcoded case_id silently rebinds to a different case when the seed data
# changes. GET /api/eval/scenarios exists precisely to avoid that.
# --------------------------------------------------------------------------
if RESET_BEFORE_RUN:
    # Drops and recreates every table, restoring deterministic seed state.
    reset = api("POST", "/api/dev/reset", role="test_operator")
    show("POST /api/dev/reset", reset)

SCENARIOS = {s["scenario_id"]: s for s in api("GET", "/api/eval/scenarios",
                                              role="eval_reader")}

seed_versions = {s["seed_version"] for s in SCENARIOS.values()}
if seed_versions != {EXPECTED_SEED_VERSION}:
    raise RuntimeError(
        f"Seed version mismatch: application reports {seed_versions}, the goldens in "
        f"this notebook were authored against {EXPECTED_SEED_VERSION!r}.\n"
        f"A golden authored against different seed data is not a weaker test, it is a "
        f"wrong one - fix the seed or the golden rather than lowering the threshold."
    )

for sid, s in sorted(SCENARIOS.items()):
    print(f"{sid}: case_id={s['case_id']} customer_id={s['customer_id']} "
          f"transaction_id={s['transaction_id']}  {s['title']}")

if not RESET_BEFORE_RUN:
    print()
    print("NOTE: AML_RESET_BEFORE_RUN is false. The tool planner skips any tool that "
          "already ran successfully\n      for this case in an earlier run, so the "
          "expected plan below is only valid on the\n      first investigation of this "
          "case after a reset. If this notebook reports missing\n      tools, set "
          "AML_RESET_BEFORE_RUN=true and re-run.")

In [ ]:
# --------------------------------------------------------------------------
# The exact request.
#
# This endpoint reads no request body: the case_id in the path is the entire
# input and all context is loaded server-side.
# --------------------------------------------------------------------------
CASE_ID = SCENARIOS["s3"]["case_id"]   # "Possible sanctions name match (beneficiary near-miss)"

print("POST", f"{API_BASE}/api/cases/{CASE_ID}/investigate")
print("headers:", json.dumps(redact({"X-API-Key": API_KEYS["analyst"] or None,
                                     "Content-Type": "application/json"}), indent=2))
print("body: (none)")

In [ ]:
# --------------------------------------------------------------------------
# The raw responses: the investigation, then the same run reshaped into
# DeepEval's LLMTestCase field names by GET /api/eval/export/{run_id}.
# --------------------------------------------------------------------------
investigation = api("POST", f"/api/cases/{CASE_ID}/investigate", expect_status=201)
RUN_ID = investigation["run_id"]
print(f"run_id = {RUN_ID}\n")

export = api("GET", f"/api/eval/export/{RUN_ID}", role="eval_reader")

show("GET /api/eval/export/{run_id}",
     {k: v for k, v in export.items() if k not in ("retrieval_context", "metadata")})
print()
print("tools_called, in DeepEval ToolCall field names, straight from the API:")
for entry in export["tools_called"]:
    print(f"  name             : {entry['name']}")
    print(f"  input_parameters : {json.dumps(entry['input_parameters'])}")
    print(f"  output           : {json.dumps(entry['output'])[:180]}")
    print()

## Mapping the API response onto DeepEval fields

| DeepEval field | API field | Note |
|---|---|---|
| `input` | `input` from the eval export | The task statement the arguments are judged against |
| `actual_output` | `actual_output` | Not scored, but supplied - the judge's reason text is more useful when it can see what the calls led to |
| `tools_called` | `tools_called[]` | `name`, `input_parameters` and `output` map one-to-one |

`expected_tools` is not set. This metric does not compare against a plan.

In [ ]:
# --------------------------------------------------------------------------
# Deterministic pre-check, before the judge is involved.
#
# GET /api/mcp/tools serves each tool's JSON Schema straight from the MCP
# protocol - the servers' own declaration, not a copy that can drift. Every
# observed argument set is validated against it here.
#
# This is a stronger and cheaper check than the judge for anything structural:
# a missing required field or a wrong type is a definite defect, whereas the
# judge is assessing semantic appropriateness. Both are reported.
# --------------------------------------------------------------------------
from jsonschema import Draft202012Validator

mcp_tools = api("GET", "/api/mcp/tools", role="eval_reader")
SCHEMAS = {t["namespaced_name"]: t["input_schema"]
           for server in mcp_tools for t in server["tools"]}

schema_failures = []
print("schema validation of every observed call:")
for entry in export["tools_called"]:
    schema = SCHEMAS.get(entry["name"])
    if schema is None:
        schema_failures.append(f"{entry['name']} is not exposed by any MCP server")
        print(f"  UNKNOWN TOOL  {entry['name']}")
        continue
    errors = sorted(Draft202012Validator(schema).iter_errors(entry["input_parameters"]),
                    key=lambda e: e.path)
    if errors:
        for error in errors:
            schema_failures.append(f"{entry['name']}: {error.message}")
            print(f"  INVALID       {entry['name']:<40} {error.message}")
    else:
        print(f"  valid         {entry['name']:<40} "
              f"{json.dumps(entry['input_parameters'])}")

print()
if schema_failures:
    print("SCHEMA VIOLATIONS FOUND - the judge score below is secondary to these:")
    for failure in schema_failures:
        print(f"  - {failure}")
else:
    print("All observed arguments validate against the tools' own JSON Schemas.")
    print("Structural correctness is established; the metric now assesses semantic "
          "appropriateness.")

In [ ]:
# --------------------------------------------------------------------------
# Debug context: the facts the arguments should have been built from.
#
# Not passed to the metric. It is here so a low score can be checked against
# reality - "screened the wrong party" is only visible if you know who the
# parties are.
# --------------------------------------------------------------------------
case = api("GET", f"/api/cases/{CASE_ID}")
customer = api("GET", f"/api/customers/{case['customer_id']}")
beneficiary = None
if case.get("transaction") and case["transaction"].get("beneficiary_id"):
    beneficiary = api("GET", f"/api/beneficiaries/{case['transaction']['beneficiary_id']}")

print("parties on this case")
print(f"  customer    : {customer['name']!r} type={customer['type']!r} "
      f"country={customer['country']!r} dob_or_inc={customer.get('incorporation_or_dob')!r}")
if beneficiary:
    print(f"  beneficiary : {beneficiary['name']!r} country={beneficiary['country']!r} "
          f"identifiers={beneficiary.get('identifiers')}")
print()
print("argument sanity checks that a reader can verify by eye:")
names_screened = [e["input_parameters"].get("name") for e in export["tools_called"]
                  if e["name"] == "risk_screening.screen_sanctions_pep"]
countries_checked = [e["input_parameters"].get("country_code") for e in export["tools_called"]
                     if e["name"] == "risk_screening.check_high_risk_country"]
print(f"  names screened   : {names_screened}")
print(f"  countries checked: {countries_checked}")
if beneficiary and countries_checked:
    print(f"  -> beneficiary country is {beneficiary['country']!r}; "
          f"customer country is {customer['country']!r}")
for entry in export["tools_called"]:
    if entry["name"] == "risk_screening.screen_sanctions_pep":
        dob = entry["input_parameters"].get("dob")
        who = entry["input_parameters"].get("name")
        print(f"  -> screened {who!r} with dob={dob!r}")

In [ ]:
# --------------------------------------------------------------------------
# Build the test case and print each DeepEval role explicitly.
# --------------------------------------------------------------------------
from deepeval.test_case import LLMTestCase, ToolCall

actual_tools = [
    ToolCall(name=entry["name"],
             input_parameters=entry["input_parameters"],
             output=entry["output"])
    for entry in export["tools_called"]
]

test_case = LLMTestCase(
    input=export["input"],
    actual_output=export["actual_output"],
    tools_called=actual_tools,
)

print("USER INPUT")
print(" ", test_case.input)
print()
print("ACTUAL TOOLS CALLED / ACTUAL ARGUMENTS")
for tool in test_case.tools_called:
    print(f"  {tool.name}")
    print(f"      arguments : {json.dumps(tool.input_parameters)}")
    print(f"      result    : {json.dumps(tool.output)[:200]}")
print()
print("EXPECTED TOOLS / EXPECTED ARGUMENTS : not used by this metric")
print("  (see ToolCorrectnessMetric.ipynb, which compares against a derived plan)")

## Judge and threshold

- **Judge model**: `DEEPEVAL_JUDGE_MODEL`, default `gpt-5.4-mini`.
- **Threshold**: `0.5`, DeepEval's documented default for `ArgumentCorrectnessMetric`.

The default is kept. This metric produces a single holistic score over all the calls in a
run, so the granularity is coarse and a high threshold would mostly measure judge mood. The
deterministic schema check above is the part of this notebook that should gate a build; the
judge score adds the semantic layer that a schema cannot express - that the *right party's*
name was screened, not merely that a string was passed where a string was required.

In [ ]:
from deepeval.metrics import ArgumentCorrectnessMetric

metric = ArgumentCorrectnessMetric(
    threshold=0.5,          # DeepEval's documented default
    model=JUDGE_MODEL,
    include_reason=True,
    async_mode=False,
    verbose_mode=True,
)
print(f"metric class : {type(metric).__name__}")
print(f"judge model  : {JUDGE_MODEL}")
print(f"threshold    : {metric.threshold}")
print(f"async_mode   : {metric.async_mode}")
print(f"strict_mode  : {metric.strict_mode}")

In [ ]:
# --------------------------------------------------------------------------
# Run the metric. A judge failure is caught and explained rather than left as
# a bare traceback, because "the judge could not be reached" and "the
# application scored badly" are completely different findings.
# --------------------------------------------------------------------------
try:
    metric.measure(test_case)
except Exception as exc:                      # noqa: BLE001 - diagnostic wrapper
    message = str(exc)
    print(f"METRIC EXECUTION FAILED: {type(exc).__name__}: {message[:600]}")
    if "api_key" in message.lower() or "authentication" in message.lower():
        print("  -> OPENAI_API_KEY is missing or rejected. This is the judge's key, "
              "not the application's.")
    elif "model" in message.lower() and "not" in message.lower():
        print(f"  -> The judge model {JUDGE_MODEL!r} was rejected. Check that your "
              f"OpenAI account can reach it, and that the installed DeepEval version "
              f"knows the id. Set DEEPEVAL_JUDGE_MODEL to change it.")
    elif "rate" in message.lower():
        print("  -> Rate limited by the judge provider. Re-run the cell.")
    raise

In [ ]:
# --------------------------------------------------------------------------
# Score, verdict, reason and debug output.
#
# Read `metric.is_successful()`, never the raw score: DeepEval metrics do not
# all point the same way. AnswerRelevancy and ToolCorrectness are "higher is
# better"; Bias and Hallucination are rates where lower is better; PIILeakage
# is a privacy score where 0.0 means maximum leakage. is_successful() applies
# the correct comparison for the metric.
# --------------------------------------------------------------------------
print(f"metric          : {type(metric).__name__}")
print(f"judge model     : {JUDGE_MODEL}")
print(f"threshold       : {metric.threshold}")
print(f"score           : {metric.score}")
print(f"PASS / FAIL     : {'PASS' if metric.is_successful() else 'FAIL'}")
print(f"judge cost (USD): {metric.evaluation_cost}")
print()
print("reason:")
print(textwrap.fill(str(metric.reason), width=96, subsequent_indent="  "))
print()
print("----- verbose judge log (debug) -----")
print(metric.verbose_logs or "(none - construct the metric with verbose_mode=True)")

## Limitations in a black-box acceptance test

1. **One score for the whole run.** A run with five calls gets a single number, so a
   single bad argument set is diluted by four good ones. Scoring per call needs one test
   case per call, which loses the cross-call context ("both parties were screened").
2. **The judge has no independent source of truth.** It sees the task statement and the
   calls. It does not know that the beneficiary's country is `TR` unless that appears in
   the task text, so it can only assess plausibility, not correctness. The debug cell above
   surfaces the real party data for a human; the deterministic checks in
   `ToolCorrectnessMetric.ipynb` are what actually verify the values.
3. **Schema validity and semantic correctness are different failures with the same
   symptom.** `{"name": "Harbour Freight Services Ltd"}` is schema-valid whether or not
   that is the party who should have been screened. Both checks are reported separately
   here for that reason.
4. **Argument construction is deterministic in this application.** The planner is
   rule-based, so this metric is really auditing a rule engine rather than a model's
   judgement. It will look stable and uninformative until those rules change - which is
   also exactly when it earns its place.
5. **Omission is invisible.** A tool that was never called has no arguments to score. Only
   `ToolCorrectnessMetric`, with an expected plan, detects a missing call.